# Model interpretowalny: binning i WoE, selekcja cech, regresja logistyczna (karta scoringowa)

----
## 0. Importy

In [1]:
import sys, json, warnings; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from src import config as cfg
from src.data import load_data, load_splits
from src.features import build_base_frame
from src.woe import WoEBinner
from src.metrics import metrics, bootstrap_ci
from src.models import cross_validate_oof, summarize_folds, save_predictions
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.style.use('default')

----
## 1. Dane i kandydaci

In [2]:
df = load_data(); splits = load_splits()
feats = json.loads((cfg.TAB_DIR / '02_cechy.json').read_text())
X = build_base_frame(df)[feats['numeric'] + feats['categorical']]
y = df[cfg.TARGET]
tr, va = splits['train'], splits['valid']
X_tr, y_tr = X.loc[tr], y.loc[tr]
print('kandydaci:', X.shape[1], '| cechy z dopuszczonym kształtem U:', cfg.U_SHAPE)

kandydaci: 129 | cechy z dopuszczonym kształtem U: ['wsk_kapital_do_aktywa', 'wsk_udzial_kap_wlasnego_aktywa_1', 'wsk_udzial_kap_wlasnego_aktywa_2', 'wsk_struktura_kap_wlasnego_2', 'wsk_struktura_kap_wlasnego_s_1', 'wsk_mnoznik_kap_wl', 'wsk_ogolnego_zadluzenia_2', 'wsk_struktury_kapitalu']


Dla 8 wskaźników struktury kapitału (`cfg.U_SHAPE`) dopuszczamy w binningu kształt U zamiast monotonicznego.

----
## 2. Binning i WoE

In [3]:
wb_all = WoEBinner(u_shape=cfg.U_SHAPE).fit(X_tr, y_tr)
iv = pd.Series(wb_all.iv_, name='IV').sort_values(ascending=False)
print('IV >= 0,02:', (iv >= 0.02).sum(), 'z', len(iv))
print(iv.head(10).round(3).to_string())
wb_all.tables_['wsk_kapital_do_aktywa'][['przedział', 'N', 'stopa złych', 'WoE', 'IV']].round(3)

IV >= 0,02: 119 z 129
wsk_struktury_kapitalu              0.411
wsk_kapital_do_aktywa               0.402
wsk_struktura_kap_wlasnego_s_1      0.397
wsk_struktura_kap_wlasnego_2        0.394
wsk_udzial_kap_wlasnego_aktywa_1    0.394
wsk_mnoznik_kap_wl                  0.385
wsk_ogolnego_zadluzenia_2           0.385
wsk_zysk_zobowiazania               0.307
wsk_struktura_kap_obcego_s          0.303
wsk_stopa_zysku_sprzedaz            0.302


,przedział,N,stopa złych,WoE,IV
0,[-inf; 0.2486),412,0.265,-0.811,0.169
1,[0.2486; 0.3855),207,0.164,-0.215,0.005
2,[0.3855; 0.7496),618,0.055,1.000,0.204
3,[0.7496; 0.8564),206,0.097,0.378,0.012
4,[0.8564; inf),618,0.133,0.042,0.001
5,brak,38,0.237,-0.697,0.011


----
## 3. Pula cech (redukcja korelacji)

In [4]:
W_tr = wb_all.transform(X_tr)
pool = []
for c in iv.index:
    if iv[c] < 0.02:
        break
    if all(abs(np.corrcoef(W_tr[c], W_tr[p])[0, 1]) <= 0.6 for p in pool):
        pool.append(c)
print(len(pool), 'cech w puli:', pool)

31 cech w puli: ['wsk_struktury_kapitalu', 'wsk_zysk_zobowiazania', 'wsk_stopa_zysku_sprzedaz', 'wsk_poziom_kapitalu_obrotowego_netto', 'wsk_ebitda_3', 'wsk_rotacja_wartosci_niewaterialnych', 'wsk_zadluzenia_krotkie', 'wsk_sytuacji_fin', 'wsk_plynnosc_biez_1', 'wsk_fin_majatku_kapitalem', 'wsk_rent_kapitalu', 'wsk_sprzedaz_kap_obrotowy', 'wsk_plynnosc_szybka_2', 'wsk_pokrycie_aktywow_kap_stalym', 'wsk_struktury_finansowania', 'wsk_zadluzenia_gotowki_1', 'wsk_pokrycia_zobowiazan_rz_aktywami_trwalymi', 'wsk_udzial_zapasy_zobowiazania', 'wsk_efekt_dzwigni_fin_2', 'wsk_rotacja_naleznosci', 'wsk_plynnosc_gotowkowa_1', 'wsk_udzial_zob_biez_sprzedaz_1', 'wsk_Zobowiazania_krotkoterminowe', 'wsk_zob_oprocentowanych_aktywa_rzeczowe', 'wsk_amortyzacja', 'wsk_zast_kapitalu_podstawowego', 'wsk_zysk_ebitda_3', 'wsk_liczba_dni_istnienia', 'formaWlasnosci_Symbol', 'schemat_wsk_rzis', 'pkd_sekcja']


Przechodzimy po cechach od najwyższego IV i dodajemy cechę tylko wtedy, gdy |ρ| jej WoE z każdą cechą z puli ≤ 0,6. Ze 119 zostaje 31 cech.

----
## 4. Selekcja krokowa na CV

In [ ]:
fold_woe = []
for f in splits['folds']:
    b = WoEBinner(u_shape=cfg.U_SHAPE).fit(X.loc[f['train'], pool], y.loc[f['train']])
    fold_woe.append((b.transform(X.loc[f['train'], pool]), y.loc[f['train']],
                     b.transform(X.loc[f['val'], pool]), y.loc[f['val']]))

def cv_auc(cols):
    aucs, signs_ok = [], True
    for w_fit, y_fit, w_val, y_val in fold_woe:
        lr = LogisticRegression(C=1e6, max_iter=2000).fit(w_fit[cols], y_fit)
        signs_ok &= bool((lr.coef_ < 0).all())        # WoE > 0 = niższe ryzyko -> oczekujemy β < 0
        aucs.append(roc_auc_score(y_val, lr.predict_proba(w_val[cols])[:, 1]))
    return np.mean(aucs), np.std(aucs), signs_ok

selected, path = [], []
for step in range(12):
    scores = {c: cv_auc(selected + [c]) for c in pool if c not in selected}
    scores = {c: s for c, s in scores.items() if s[2]}          # tylko kandydaci z poprawnymi znakami
    best = max(scores, key=lambda c: scores[c][0])
    selected.append(best)
    path.append({'k': step + 1, 'dodana cecha': best, 'AUC CV': scores[best][0], 'odch. std': scores[best][1]})
path = pd.DataFrame(path).set_index('k')
k_best = path.loc[8:12, 'AUC CV'].idxmax()                      # decyzja: 8-12 cech, max AUC na CV
SELECTED = selected[:k_best]
print('wybrana liczba cech:', k_best); path.round(4)

**Wnioski:** AUC na CV rośnie do 9 cech (0,718), potem lekko spada. Kolejne cechy nie poprawiają modelu, a różnice są dużo mniejsze od odchylenia między foldami (≈ 0,03). Wybieramy 9 cech; wszystkie mają poprawny znak β na każdym foldzie.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.errorbar(path.index, path['AUC CV'], yerr=path['odch. std'], marker='o', capsize=3)
ax.axvline(k_best, ls='--', c='grey'); ax.set_xlabel('liczba cech'); ax.set_ylabel('AUC (średnia z 5 foldów)')
ax.set_title('Selekcja krokowa: AUC na CV ± odch. std'); ax.set_xticks(path.index)
plt.tight_layout(); plt.savefig(cfg.FIG_DIR / '03_selekcja.png', dpi=150); plt.show()

----
## 5. Model końcowy i współczynniki

In [ ]:
logit_model = Pipeline([('woe', WoEBinner(u_shape=cfg.U_SHAPE)),
                        ('lr', LogisticRegression(C=1e6, max_iter=2000))])
logit_model.fit(X_tr[SELECTED], y_tr)
W_sel = logit_model.named_steps['woe'].transform(X_tr[SELECTED])
A = np.column_stack([np.ones(len(W_sel)), W_sel.values])
coef = pd.DataFrame({
    'β': logit_model.named_steps['lr'].coef_[0],
    'VIF': [variance_inflation_factor(A, i + 1) for i in range(len(SELECTED))],
    'IV': [logit_model.named_steps['woe'].iv_[c] for c in SELECTED],
    'liczba binów': [len(logit_model.named_steps['woe'].tables_[c]) for c in SELECTED],
}, index=SELECTED)
coef['znak OK (β<0)'] = coef['β'] < 0
print('wyraz wolny:', round(logit_model.named_steps['lr'].intercept_[0], 3))
coef.to_csv(cfg.TAB_DIR / '03_wspolczynniki.csv')
coef.round(3)

**Wnioski:** wszystkie β < 0 (wyższe WoE = niższe ryzyko, zgodnie z oczekiwaniem), VIF < 2 (brak współliniowości), każda cecha ma 3–5 przedziałów (razem 38). Model ma 10 parametrów i da się go w całości zapisać jako kartę punktową.

----
## 6. Walidacja krzyżowa (OOF)

In [ ]:
oof_logit, folds_logit = cross_validate_oof(logit_model, X[SELECTED], y, splits)
folds_logit.to_csv(cfg.TAB_DIR / '03_cv_foldy.csv')
summarize_folds(folds_logit).round(4)

**Wnioski:** AUC na 5 foldach 0,718 ± 0,034 (Gini 0,44). Rozrzut między foldami jest duży względem różnic między modelami, co trzeba brać pod uwagę przy porównaniach.

----
## 7. Wyniki na train / OOF / valid

In [ ]:
p_tr = logit_model.predict_proba(X_tr[SELECTED])[:, 1]
p_va = logit_model.predict_proba(X.loc[va, SELECTED])[:, 1]
res = pd.DataFrame({'train': metrics(y_tr, p_tr), 'OOF (CV)': metrics(y.loc[oof_logit.index], oof_logit),
                    'valid': metrics(y.loc[va], p_va)}).T
res.to_csv(cfg.TAB_DIR / '03_metryki.csv')
display(res.drop(columns='n').round(4))
bootstrap_ci(y.loc[va], p_va).round(3)

**Wnioski:** różnica train–OOF jest mała (AUC 0,749 vs 0,716), więc model nie jest przeuczony. Na valid AUC = 0,748, ale 95% przedział ufności jest szeroki [0,680; 0,814], bo valid ma tylko 62 defaulty. Kalibracja jest dobra (ECE 1,7% na OOF, 4,4% na valid).

----
## 8. Profile WoE

In [ ]:
n = len(SELECTED); ncol = 4; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3.6 * nrow))
for ax, c in zip(axes.ravel(), SELECTED):
    t = logit_model.named_steps['woe'].tables_[c]
    ax.bar(range(len(t)), t['WoE'], color=['#C44E52' if w < 0 else '#4C72B0' for w in t['WoE']])
    ax.set_xticks(range(len(t))); ax.set_xticklabels(t['przedział'], rotation=60, ha='right', fontsize=7)
    ax.axhline(0, c='k', lw=0.8); ax.set_title(f'{c}' + (' (U)' if c in cfg.U_SHAPE else ''), fontsize=9)
    ax.set_ylabel('WoE  (<0 = wyższe ryzyko)', fontsize=8)
for ax in axes.ravel()[n:]:
    ax.axis('off')
plt.tight_layout(); plt.savefig(cfg.FIG_DIR / '03_profile_woe.png', dpi=150); plt.show()

**Wnioski:** profile pokazują, jak zmienia się ryzyko w przedziałach każdej cechy (czerwone = wyższe ryzyko od średniej). Kierunki zgadzają się z intuicją ekonomiczną, np. niższa rentowność i wyższe zadłużenie krótkoterminowe to większe ryzyko.

----
## 9. Karta punktowa

In [ ]:
PDO, BASE_SCORE, BASE_ODDS = 20, 600, 50
factor = PDO / np.log(2); offset = BASE_SCORE - factor * np.log(BASE_ODDS)
b0 = logit_model.named_steps['lr'].intercept_[0]; betas = dict(zip(SELECTED, logit_model.named_steps['lr'].coef_[0]))
rows = []
for c in SELECTED:                              # log-szanse DOBREGO = -logit(PD) -> punkty rosną z WoE
    for _, r in logit_model.named_steps['woe'].tables_[c].iterrows():
        rows.append({'cecha': c, 'przedział': r['przedział'], 'WoE': r['WoE'],
                     'punkty': round(factor * (-betas[c] * r['WoE'] - b0 / len(SELECTED)) + offset / len(SELECTED))})
scorecard = pd.DataFrame(rows)
scorecard.to_csv(cfg.TAB_DIR / '03_karta_punktowa.csv', index=False)
print(f'Factor = {factor:.3f}, Offset = {offset:.2f}')
scorecard[scorecard['cecha'] == SELECTED[0]]

**Wnioski:** skala PDO 20 i 600 pkt przy szansach 50:1. Punkty rosną z WoE, czyli więcej punktów = mniejsze ryzyko. Dla `wsk_struktury_kapitalu` firma dostaje 47–77 pkt, a najwięcej w przedziale [1,0; 2,9) (kształt U).

----
## 10. Zapis wyników

In [ ]:
p_te = logit_model.predict_proba(X.loc[splits['test'], SELECTED])[:, 1]   # zapisujemy, NIE oceniamy (krok 08)
save_predictions('logit', oof_logit, p_va, p_te, splits)
(cfg.ROOT / 'outputs' / 'models').mkdir(parents=True, exist_ok=True)
joblib.dump({'model': logit_model, 'features': SELECTED}, cfg.ROOT / 'outputs' / 'models' / 'logit.joblib')
(cfg.TAB_DIR / '03_cechy_logit.json').write_text(json.dumps({'pool': pool, 'selected': SELECTED}, indent=1))
print('zapisano: outputs/predictions/logit.csv, outputs/models/logit.joblib, outputs/tables/03_cechy_logit.json')

**Wnioski:** predykcje na teście zapisujemy, ale ich jeszcze nie oceniamy – test jest używany jednorazowo w notebooku 08.